In [1]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

# from marc21_parser import Marc21Parser --- IGNORE ---
from core.cleaning import Cleaner
from core.classification_transform import ClassificationTransformer
from core.data_explorer import Marc21Explorer

In [ ]:
#Chart 1 - über die Zeit: Anzahl der Dissertationen pro Dekade, gefärbt nach DDC-Hauptklasse → zeigt die Entwicklung der DDC-Abdeckung über die Zeit
#Chart 2 - Sunburst-Chart der DDC-Hauptklassen und ihrer Top-Subklassen → zeigt die Hierarchie der DDC und die Verteilung innerhalb der Hauptklassen
#Chart 3 - Systemwechsel über Zeit
#Chart 4 — Mineralogie-Records nach Jahrzehnt, gefärbt nach Klassifikationsquelle → zeigt den Testfall konkret
#Chart 5 — Unklassifizierte Records nach Jahrzehnt → das ist dein Retro-Argument, visuell
#Chart 6 — Top-DDC-549-Subklassen → zeigt was die DNB unter Mineralogie versteht
#Chart 7 — 

In [2]:
# df = pd.read_parquet("../data/raw/dnb-all_hochschulschriften_dnbmarc.mrc.xml.gz")
#--------- oder Rohdaten aus Parquet einlesen, wenn Pfad vorhanden
parquet_file = "../data/raw/dnb_all_theses.parquet"
df_raw = pd.read_parquet(parquet_file)

In [ ]:
###ÜBERARBEITEN 
# 
# 
# df_clean = Cleaner.clean_library_df(df_raw)

# Optional Transformer
SDNB_TO_DDC = {
    "01": "000", "02": "100", "05": "500", "06": "600",
    "07": "700", "08": "800", "09": "900"
}


transformer = ClassificationTransformer(sdnb_to_ddc_mapping=SDNB_TO_DDC)
df_final = transformer.apply(df_clean)


explorer = Marc21Explorer(df_final)

print(df_final.shape)
df_final.head()

UFuncTypeError: ufunc 'add' did not contain a loop with signature matching types (dtype('float64'), dtype('<U2')) -> None

In [ ]:
# Plot 1 Kalassifizierung des Gesamtdatensets - dem entsprechend Subsets gebildet vom DNBLab

# Plotly Sunburst-Chart erfordert keine explizite Hierarchie, aber wir können die Hauptklassen benennen

DDC_MAIN = {
    "0": "Allgemeines",
    "1": "Philosophie",
    "2": "Religion",
    "3": "Sozialwissenschaften",
    "4": "Sprache",
    "5": "Naturwissenschaften",
    "6": "Technik",
    "7": "Kunst",
    "8": "Literatur",
    "9": "Geschichte"
}

# Datenbasis prüfen: Wie viele echte DDC_1_label haben wir?
num_real_records = df_ddc.dropna(subset=["DDC_1"]).shape[0]
print(f"Echte Datensätze mit DDC_1: {num_real_records}")


# Ersetze fehlende Werte mit "Unbekannt"
for col in ["DDC_1", "DDC_2", "DDC_3"]:
    df_ddc[col] = df_ddc[col].fillna("Unbekannt")


fig = px.sunburst(
    df_ddc,
    path=["DDC_1", "DDC_2", "DDC_3"],
    title="DDC-Hierarchie mit Hauptklassen"
)
fig.show()


In [ ]:
# Plot 2: Mehrere Spalten pro Dekade mit Plotly

def plot_multi_columns_plotly(df, columns):
    data = df.copy()
    data = data[pd.to_numeric(data["publication_year"], errors="coerce").notna()]
    data["publication_year"] = data["publication_year"].astype(int)
    data["decade"] = (data["publication_year"] // 10) * 10

    total_per_decade = data.groupby("decade").size()
    decades = sorted(total_per_decade.index)

    percent_matrix = pd.DataFrame(index=decades)

    for col in columns:
        if col == "has_sdnb":
            presence = data.groupby("decade")[col].apply(lambda s: (s == True).sum())
        elif col == "sdnb_codes":
            presence = data.groupby("decade")[col].apply(
                lambda s: s.apply(lambda x: isinstance(x, list) and len(x) > 0).sum()
            )
        else:
            presence = data.groupby("decade")[col].apply(lambda s: s.notna().sum())

        percent_matrix[col] = (presence / total_per_decade * 100).fillna(0)

    fig = go.Figure()

    for col in columns:
        fig.add_trace(go.Bar(
            x=[str(d) for d in decades],
            y=percent_matrix[col],
            name=col
        ))

    fig.update_layout(
        title="Qualität pro Dekade",
        xaxis_title="Dekade",
        yaxis_title="Prozent mit Inhalt",
        yaxis=dict(range=[0, 100]),
        barmode="group",
        template="plotly_white"
    )

    fig.show()

    return percent_matrix


plot_multi_columns_plotly(df_raw, [
    "author_gnd",
    "dissertation_note",
    "has_sdnb",
    "sdnb_codes"
])

ValueError: invalid literal for int() with base 10: '1956.'

In [ ]:
# Plot3: Heatmap + Top-Mappings
explorer.analyze_sdnb_ddc_plotly(top_n=10)

In [ ]:
# Plot 4: SDNB-Abdeckung über Zeit
# Anteil SDNB pro Dekade
df_final["decade"] = (df_final["publication_year"] // 10) * 10

sdnb_over_time = (
    df_final.groupby("decade")["has_sdnb"]
    .mean()
    .reset_index()
)

fig = px.bar(
    sdnb_over_time,
    x="decade",
    y="has_sdnb",
    labels={"has_sdnb": "Anteil SDNB"},
    title="SDNB-Abdeckung über Zeit"
)

fig.update_layout(yaxis=dict(range=[0, 1]))
fig.show()

In [ ]:
# Plot 5: DDC-Hauptklassenverteilung
ddc_dist = df_final["ddc_main_class"].value_counts().reset_index()
ddc_dist.columns = ["ddc_main_class", "count"]

fig = px.bar(
    ddc_dist,
    x="ddc_main_class",
    y="count",
    title="Verteilung der DDC-Hauptklassen"
)

fig.show()

In [ ]:
# Beispiel: wie viele SDNB vorhanden?
print(df_final["has_sdnb"].value_counts(normalize=True) * 100)

# Listenfeld-Stats
explorer.field_stats("sdnb_codes")

In [ ]:
## Plot 6